# 7. Grad-CAM 시각화

이 노트북은 `06_Transfer_Learning_실험.ipynb` 다음 단계로, **CNN이 이미지의 어느 부분을 보고 판단했는지** 를 시각적으로 확인하는 실습입니다.

앞선 노트북에서는 pretrained ResNet-18을 이용해 transfer learning을 수행했습니다. 하지만 정확도 숫자만으로는 모델이 정말 올바른 근거를 보고 판단했는지 알기 어렵습니다. 이번에는 `Grad-CAM`을 사용해, 모델이 특정 클래스를 예측할 때 **어느 위치에 주목했는지** heatmap 형태로 확인해 보겠습니다.

이번 노트북의 목표는 다음과 같습니다.

- Grad-CAM의 핵심 아이디어를 이해합니다.
- ResNet-18의 마지막 convolution stage에서 activation과 gradient를 추출합니다.
- 예측 결과 위에 heatmap을 덧씌워, 모델이 어떤 부분을 보고 판단했는지 확인합니다.
- 올바르게 맞춘 예시와 틀린 예시를 비교하며 해석해 봅니다.


## 7-1. Grad-CAM은 무엇일까?

Grad-CAM(Gradient-weighted Class Activation Mapping)은 특정 클래스 점수에 대해, 마지막 convolution feature map의 각 채널이 얼마나 중요한지 gradient로 계산한 뒤, 이를 가중합해 시각화하는 방법입니다.

흐름을 간단히 요약하면 다음과 같습니다.

- 모델이 어떤 클래스 점수를 출력합니다.
- 그 점수를 기준으로 마지막 convolution feature map에 대한 gradient를 구합니다.
- gradient를 공간 평균해서 채널별 중요도를 구합니다.
- 중요도로 feature map을 가중합하고, 양수 부분만 남겨 heatmap을 만듭니다.

즉, Grad-CAM은 **"이 예측을 만들 때 어떤 위치 정보가 특히 중요했는가?"** 를 보여 주는 방법입니다.


## 7-2. 준비

이번 노트북은 pretrained ResNet-18을 기준으로 진행합니다. 학습을 길게 다시 하지 않고도 Grad-CAM 흐름을 확인할 수 있도록, 기본값은 `pretrained weights`를 사용하고 마지막 분류기만 CIFAR-10용으로 바꾸는 형태로 두었습니다.

엄밀한 성능 실험보다 해석 흐름 이해가 목적이므로, 기본 학습 epoch는 작게 두었습니다. 이미 더 잘 학습된 가중치가 있다면 그 모델을 불러오도록 확장해도 됩니다.


In [ ]:
# 필요 라이브러리가 없다면 아래 주석을 해제해서 설치하세요.
# !pip install torch torchvision matplotlib


In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)


## 7-3. 데이터 준비

ResNet-18 pretrained weights를 쓰기 위해 입력 이미지는 `224 x 224`로 맞춥니다. Grad-CAM 시각화를 위해 원본에 가까운 이미지를 보여 줄 수 있도록, 평가용 transform도 함께 준비합니다.


In [ ]:
image_size = 224
train_size = 4000
val_size = 1000
batch_size = 32

mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

full_train_aug = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
full_train_eval = datasets.CIFAR10(root='./data', train=True, download=False, transform=eval_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform)

classes = full_train_aug.classes

generator = torch.Generator().manual_seed(42)
all_indices = torch.randperm(len(full_train_aug), generator=generator).tolist()
train_indices = all_indices[:train_size]
val_indices = all_indices[train_size:train_size + val_size]

train_dataset = Subset(full_train_aug, train_indices)
val_dataset = Subset(full_train_eval, val_indices)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size * 2, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size * 2, shuffle=False)

print('Classes:', classes)
print('Train samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Test samples:', len(test_dataset))


In [ ]:
def denormalize(image):
    mean_tensor = torch.tensor(mean).view(3, 1, 1)
    std_tensor = torch.tensor(std).view(3, 1, 1)
    return (image.detach().cpu() * std_tensor + mean_tensor).clamp(0, 1)


images, labels = next(iter(test_loader))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(denormalize(image).permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis('off')
plt.tight_layout()
plt.show()


## 7-4. 모델 준비

Grad-CAM을 제대로 보려면 최소한 CIFAR-10에 어느 정도 적응된 모델이 필요합니다. 여기서는 pretrained ResNet-18을 불러와 마지막 `fc`만 바꾸고, 짧게 fine-tuning하는 기본 예시를 사용합니다.

이미 `06_Transfer_Learning_실험.ipynb`에서 저장한 가중치가 있다면 불러와 연결할 수도 있지만, 이 노트북은 독립적으로 실행될 수 있도록 다시 구성합니다.


In [ ]:
use_pretrained = True
run_training = True
epochs = 1
learning_rate = 0.0001


def build_model(num_classes=10, use_pretrained=True):
    weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


model = build_model(num_classes=len(classes), use_pretrained=use_pretrained)
print('Grad-CAM 대상 layer:', model.layer4[-1])


보통 ResNet에서는 마지막 convolution stage인 `layer4`를 Grad-CAM 대상으로 많이 사용합니다. 너무 앞쪽 층은 저수준 특징만 담고 있고, 너무 뒤는 spatial 정보가 거의 사라지기 때문입니다.


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, epochs=1, lr=0.0001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = []

    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history.append((train_loss, train_acc, val_loss, val_acc))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    return model, history, criterion


In [ ]:
if run_training:
    model, history, criterion = train_model(
        model,
        train_loader,
        val_loader,
        epochs=epochs,
        lr=learning_rate
    )
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    print(f'Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}')
else:
    model = model.to(device)
    print('학습을 건너뛰고 pretrained + 새 fc 상태의 모델로 진행합니다.')


## 7-5. Grad-CAM 구현

이제 핵심 구현입니다. 대상 layer의 forward activation과 backward gradient를 저장한 뒤, 이를 이용해 heatmap을 계산합니다.


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(self._save_activation)
        self.backward_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        score = output[:, class_idx]
        self.model.zero_grad()
        score.backward(retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(
            cam,
            size=input_tensor.shape[-2:],
            mode='bilinear',
            align_corners=False
        )

        cam = cam.squeeze().cpu()
        cam -= cam.min()
        cam /= cam.max() + 1e-8
        return cam, class_idx, output.detach()

    def remove_hooks(self):
        self.forward_handle.remove()
        self.backward_handle.remove()


In [ ]:
grad_cam = GradCAM(model, model.layer4[-1])
print('Grad-CAM hook 등록 완료')


## 7-6. 한 장의 이미지에 대해 Grad-CAM 보기

먼저 테스트 이미지 한 장에 대해 모델 예측과 heatmap을 확인합니다. heatmap이 강한 부분일수록 그 클래스 판단에 더 크게 기여한 위치라고 볼 수 있습니다.


In [ ]:
sample_images, sample_labels = next(iter(test_loader))
sample_index = 0

input_tensor = sample_images[sample_index:sample_index + 1].to(device)
target_label = sample_labels[sample_index].item()

cam, pred_idx, logits = grad_cam.generate(input_tensor)
probs = logits.softmax(dim=1).squeeze().cpu()

image = denormalize(input_tensor.squeeze(0)).permute(1, 2, 0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title(f'Original\nT: {classes[target_label]}')
axes[0].axis('off')

axes[1].imshow(cam, cmap='jet')
axes[1].set_title(f'Grad-CAM\nP: {classes[pred_idx]}')
axes[1].axis('off')

axes[2].imshow(image)
axes[2].imshow(cam, cmap='jet', alpha=0.45)
axes[2].set_title(
    f'Overlay\nP: {classes[pred_idx]} ({probs[pred_idx]:.3f})'
)
axes[2].axis('off')

plt.tight_layout()
plt.show()


해석할 때는 단순히 heatmap이 예쁘게 나오는지만 보면 안 됩니다. 예를 들어 `자동차`를 맞췄는데 실제로는 배경이나 도로 쪽만 강하게 보고 있다면, 모델이 편향된 단서를 이용하고 있을 가능성도 있습니다.


## 7-7. 여러 이미지 한꺼번에 시각화

이제 여러 테스트 이미지에 대해 원본, overlay, 정답/예측 정보를 함께 확인합니다. 맞춘 예시와 틀린 예시를 같이 보는 것이 중요합니다.


In [ ]:
def visualize_gradcam_batch(model, grad_cam, loader, classes, num_images=6):
    images, labels = next(iter(loader))
    images = images.to(device)
    labels = labels.to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = outputs.argmax(dim=1)

    fig, axes = plt.subplots(num_images, 2, figsize=(8, num_images * 3))
    if num_images == 1:
        axes = [axes]

    for row in range(num_images):
        image_tensor = images[row:row + 1]
        label = labels[row].item()
        pred = preds[row].item()
        cam, _, logits = grad_cam.generate(image_tensor, class_idx=pred)
        prob = logits.softmax(dim=1)[0, pred].item()
        image = denormalize(image_tensor.squeeze(0)).permute(1, 2, 0)

        axes[row][0].imshow(image)
        axes[row][0].set_title(f'Original\nT: {classes[label]}')
        axes[row][0].axis('off')

        axes[row][1].imshow(image)
        axes[row][1].imshow(cam, cmap='jet', alpha=0.45)
        axes[row][1].set_title(
            f'Overlay\nP: {classes[pred]} ({prob:.3f})'
        )
        axes[row][1].axis('off')

    plt.tight_layout()
    plt.show()


visualize_gradcam_batch(model, grad_cam, test_loader, classes, num_images=6)


여러 예시를 보다 보면 다음과 같은 패턴을 볼 수 있습니다.

- 맞춘 경우에는 물체 중심부나 윤곽 근처에 heatmap이 모이는 경우가 많습니다.
- 틀린 경우에는 배경, 색상 덩어리, 이미지 구석처럼 덜 본질적인 부분을 강하게 보는 경우가 있습니다.
- 비슷한 클래스(`cat` vs `dog`, `car` vs `truck`)는 주목 위치가 겹치지만, 세부 부위 해석이 달라질 수 있습니다.


## 7-8. 맞춘 예시와 틀린 예시 나누어 보기

Grad-CAM은 특히 틀린 예시를 분석할 때 유용합니다. 모델이 왜 헷갈렸는지, 물체보다 배경을 더 본 것은 아닌지 확인할 수 있기 때문입니다.


In [ ]:
def collect_examples(model, loader, max_examples=2):
    correct_examples = []
    wrong_examples = []
    model.eval()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            outputs = model(images)
            preds = outputs.argmax(dim=1)

        for image, label, pred in zip(images, labels, preds):
            item = (image.unsqueeze(0), label.item(), pred.item())
            if label.item() == pred.item() and len(correct_examples) < max_examples:
                correct_examples.append(item)
            if label.item() != pred.item() and len(wrong_examples) < max_examples:
                wrong_examples.append(item)
            if len(correct_examples) >= max_examples and len(wrong_examples) >= max_examples:
                return correct_examples, wrong_examples

    return correct_examples, wrong_examples


def show_example_group(title, examples, grad_cam, classes):
    if not examples:
        print(f'{title}: 해당 예시를 충분히 찾지 못했습니다.')
        return

    fig, axes = plt.subplots(len(examples), 2, figsize=(8, len(examples) * 3))
    if len(examples) == 1:
        axes = [axes]

    for row, (image_tensor, label, pred) in enumerate(examples):
        cam, _, logits = grad_cam.generate(image_tensor, class_idx=pred)
        prob = logits.softmax(dim=1)[0, pred].item()
        image = denormalize(image_tensor.squeeze(0)).permute(1, 2, 0)

        axes[row][0].imshow(image)
        axes[row][0].set_title(f'Original\nT: {classes[label]}')
        axes[row][0].axis('off')

        axes[row][1].imshow(image)
        axes[row][1].imshow(cam, cmap='jet', alpha=0.45)
        axes[row][1].set_title(
            f'Overlay\nP: {classes[pred]} ({prob:.3f})'
        )
        axes[row][1].axis('off')

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


correct_examples, wrong_examples = collect_examples(model, test_loader, max_examples=2)
show_example_group('맞춘 예시', correct_examples, grad_cam, classes)
show_example_group('틀린 예시', wrong_examples, grad_cam, classes)


## 7-9. 해석할 때 주의할 점

Grad-CAM은 매우 유용하지만, 몇 가지 한계도 함께 이해해야 합니다.

- heatmap은 어디가 중요했는지 보여 주지만, 정확한 인과 관계를 완전히 증명하지는 않습니다.
- 마지막 convolution layer를 기준으로 하므로 공간 해상도가 비교적 거칠 수 있습니다.
- 잘못 학습된 모델이라면 heatmap도 그만큼 엉뚱한 위치를 강조할 수 있습니다.
- 따라서 Grad-CAM은 "정답을 보장하는 설명" 이라기보다, **모델 해석을 도와주는 진단 도구** 로 보는 것이 적절합니다.


## 정리

이번 노트북의 핵심은 다음과 같습니다.

- Grad-CAM은 특정 클래스 예측에 중요했던 위치를 heatmap으로 보여 줍니다.
- ResNet에서는 보통 `layer4` 같은 마지막 convolution stage를 대상으로 많이 사용합니다.
- 맞춘 예시뿐 아니라 틀린 예시도 함께 보면, 모델이 왜 그런 판단을 했는지 더 잘 이해할 수 있습니다.
- 이런 시각화는 모델 해석, 오류 분석, 데이터 편향 점검에 매우 유용합니다.

다음 단계로는 `Guided Backpropagation`, `Integrated Gradients`, `Attention map 비교` 같은 다른 해석 기법도 이어서 실험해 볼 수 있습니다.
